# NB_hospital_daily_summary

Builds `<catalog>.gold.hospital_daily_summary` (one row per hospital per calendar day with lab activity) per the STTM `sttm_silver_to_gold.xlsx`. Driving table `silver.lab_results` grouped by `hospital_id, test_date`, LEFT JOINs to `hospitals` and the calendar dimension, plus a LAG() day-over-day trend. Full rebuild.

In [0]:
# <catalog> placeholder from the STTM; override via widget / job parameter
dbutils.widgets.text("catalog", "adb_newcaresync")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
print(f"Using catalog: {catalog}")

In [0]:
%sql
-- Note: STTM names the calendar source silver.calendar; in this workspace it is silver.dim_calendar
CREATE OR REPLACE TABLE gold.hospital_daily_summary
COMMENT 'Gold: one row per hospital per day with lab-test volume, abnormal/critical rates and day-over-day trend (source: silver)'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                                AS summary_date,
    COUNT(*)                                 AS total_tests,
    COUNT(DISTINCT patient_id)               AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')     AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')     AS critical_test_count
  FROM silver.lab_results
  GROUP BY hospital_id, test_date
),
trend AS (
  SELECT
    d.*,
    LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(t.hospital_id AS INT)                                                   AS hospital_id,
  h.hospital_name,
  t.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(t.total_tests AS INT)                                                   AS total_tests,
  CAST(t.unique_patients_tested AS INT)                                        AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT)                                           AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT)                                           AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                           AS critical_test_count,
  CAST(ROUND(t.abnormal_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(ROUND(t.critical_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT)                                          AS prev_day_total_tests,
  CASE WHEN t.prev_day_total_tests IS NULL OR t.prev_day_total_tests = 0 THEN NULL
       ELSE CAST(ROUND((t.total_tests - t.prev_day_total_tests) * 100.0 / t.prev_day_total_tests, 2) AS DECIMAL(6,2))
  END                                                                          AS test_volume_change_pct
FROM trend t
LEFT JOIN silver.hospitals h    ON t.hospital_id = h.hospital_id
LEFT JOIN silver.dim_calendar c ON c.date_key = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT);

In [0]:
record_count = spark.table("gold.hospital_daily_summary").count()
dbutils.notebook.exit(str(record_count))